In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import numpy as np
import os

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
val_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


In [5]:
# extracting and transforming the data using ImageFolder

train_dataset = datasets.ImageFolder(
    root = "train",
    transform = train_transform
)
test_dataset = datasets.ImageFolder(
    root = "test",
    transform = test_transform
)
val_dataset = datasets.ImageFolder(
    root = "val",
    transform = val_transform
)



In [6]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False)

In [7]:
print(train_dataset.classes)
print(test_dataset.classes)
print(len(train_dataset))
print(len(test_dataset))

['Battery', 'Keyboard', 'Microwave', 'Mobile', 'Mouse', 'PCB', 'Player', 'Printer', 'Television', 'Washing Machine']
['Battery', 'Keyboard', 'Microwave', 'Mobile', 'Mouse', 'PCB', 'Player', 'Printer', 'Television', 'Washing Machine']
2400
300


In [8]:
class EWasteCNN(nn.Module):
  def __init__(self, input_features):
    super().__init__()
    self.features = nn.Sequential( # dont apply dropout here
        nn.Conv2d(input_features, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),

        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.AdaptiveAvgPool2d((1,1))
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),

        nn.Linear(256, 64),
        nn.ReLU(),
        nn.Dropout(p =0.4),
        nn.Linear(64,10)
    )


  def forward(self, x):
    x= self.features(x)
    x=self.classifier(x)
    return x



In [9]:
model = EWasteCNN(input_features=3)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001, weight_decay=1e-4)

In [ ]:
epochs = 40
for epoch in range(epochs):

    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images
        labels = labels

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")

In [18]:
training_log = []

epochs = 20

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_acc = correct / total


    training_log.append({
        "epoch": epoch + 1,
        "train_acc": train_acc,
        "test_acc": test_acc
    })


    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

Epoch 1/20
Train Acc: 0.7575 | Test Acc: 0.7433
Epoch 2/20
Train Acc: 0.7583 | Test Acc: 0.7400
Epoch 3/20
Train Acc: 0.7688 | Test Acc: 0.7400
Epoch 4/20
Train Acc: 0.7538 | Test Acc: 0.7400
Epoch 5/20
Train Acc: 0.7550 | Test Acc: 0.7300
Epoch 6/20
Train Acc: 0.7717 | Test Acc: 0.7367
Epoch 7/20
Train Acc: 0.7621 | Test Acc: 0.7400
Epoch 8/20
Train Acc: 0.7667 | Test Acc: 0.7333
Epoch 9/20
Train Acc: 0.7608 | Test Acc: 0.7467
Epoch 10/20
Train Acc: 0.7525 | Test Acc: 0.7433
Epoch 11/20
Train Acc: 0.7608 | Test Acc: 0.7400
Epoch 12/20
Train Acc: 0.7708 | Test Acc: 0.7433
Epoch 13/20
Train Acc: 0.7562 | Test Acc: 0.7433
Epoch 14/20
Train Acc: 0.7521 | Test Acc: 0.7400
Epoch 15/20
Train Acc: 0.7562 | Test Acc: 0.7400
Epoch 16/20
Train Acc: 0.7588 | Test Acc: 0.7433
Epoch 17/20
Train Acc: 0.7612 | Test Acc: 0.7333
Epoch 18/20
Train Acc: 0.7562 | Test Acc: 0.7367
Epoch 19/20
Train Acc: 0.7529 | Test Acc: 0.7400
Epoch 20/20
Train Acc: 0.7546 | Test Acc: 0.7400


In [27]:
import json

with open("cnn_baseline_metrics.json","w") as f:
    json.dump(training_log, f, indent = 2)


In [30]:
torch.save(model.state_dict(), "ewaste_model_cnn_v2.pth")

In [31]:
import os

print(os.path.getsize("ewaste_model_cnn.pth"))

1640857


In [32]:
model = EWasteCNN(input_features=3)
model.load_state_dict(torch.load("ewaste_model_cnn.pth"))

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        images = images
        labels = labels

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Test Accuracy:", accuracy)

Test Accuracy: 72.0


In [33]:
model = EWasteCNN(input_features=3)
model.load_state_dict(torch.load("ewaste_model_cnn.pth"))

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in train_loader:

        images = images
        labels = labels

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Train Accuracy:", accuracy)

Train Accuracy: 78.70833333333333
